In [ ]:
using Pkg
Pkg.activate("/Users/bursche/Documents/GitHub/JPEC_BCRIT")
Base.active_project()

using GeneralizedPerturbedEquilibrium
using GeneralizedPerturbedEquilibrium: Analysis
using Printf

using Plots
default(
    fontfamily="Georgia",
    margin=12Plots.mm,
    size=(800, 500),
    dpi=150
)

In [ ]:
h5path = "gpec.h5"

In [ ]:
println(keys(h5open(h5path, "r")["Tearing"]["CriticalResonantField"]["Scan"]["surface_1"]))

In [ ]:
function bcrit_diag_plots(Δs, Qs, bal, name)

    p1 = plot(Qs, imag.(Δs), label="Im(Δ)", lw=2)
    plot!(p1, Qs, real.(Δs), label="Re(Δ)", lw=2)
    xlabel!(p1, "Q")
    ylabel!(p1, "Δ")
    title!(p1, "Inner-layer Δ(Q) - $name")

    p2 = plot(Qs, real.(bal), label="Re(balance)", lw=2)
    plot!(p2, Qs, imag.(bal), label="Im(balance)", lw=2)
    xlabel!(p2, "Q")
    ylabel!(p2, "balance")
    title!(p2, "2P(Q0-Q)/jxb - $name")

    plot(p1, p2, layout=(2,1), size=(500, 700))

end

function plot_all_vs_rational_q(h5path)
    h5open(h5path, "r") do file
        tearing = file["Tearing"]
        crf = tearing["CriticalResonantField"]
        scan = crf["Scan"]

        rational_q = read(tearing["PerSurface"]["rational_q"])
        bcrit = read(crf["br_crit"])
        bal = [read(scan["surface_$(i)"]["balance"]) for i in eachindex(rational_q)]
        P = [read(scan["surface_$(i)"]["P"]) for i in eachindex(rational_q)]
        lu = [read(scan["surface_$(i)"]["lu"]) for i in eachindex(rational_q)]
        sval = [read(scan["surface_$(i)"]["sval"]) for i in eachindex(rational_q)]
        Q0 = [read(scan["surface_$(i)"]["Q0"]) for i in eachindex(rational_q)]
        max_balance = maximum.(bal)

        fig = plot(layout=(3, 2), size=(1000, 1200))
        plot!(fig[1], rational_q, bcrit; marker=:circle, lw=2, xlabel="Rational Surface (q)", ylabel="Critical Resonant Field (T)", legend=false)
        plot!(fig[2], rational_q, max_balance; marker=:circle, lw=2, xlabel="Rational Surface (q)", ylabel="Maximum (Torque Balance)", legend=false)
        plot!(fig[3], rational_q, P; marker=:circle,  lw=2, xlabel="Rational Surface (q)", ylabel="P", legend=false)
        plot!(fig[4], rational_q, lu; marker=:circle, lw=2, xlabel="Rational Surface (q)", ylabel="Lundquist Number", legend=false)
        plot!(fig[5], rational_q, sval; marker=:circle, lw=2, xlabel="Rational Surface (q)", ylabel="Magnetic Shear", legend=false)
        plot!(fig[6], rational_q, Q0; marker=:circle, lw=2, xlabel="Rational Surface (q)", ylabel="Q0", legend=false)

        display(fig)
        #return fig
    end
end

In [ ]:
plot_all_vs_rational_q(h5path)

In [ ]:
# Plot All Inner-layer Δ(Q) and Torque Balance vs Q for each rational surface

h5open(h5path, "r") do file
    bcrit = file["Tearing"]["CriticalResonantField"]

    for i in 1:6
        ss = "surface_$i"

        delta  = read(bcrit["Scan"][ss]["delta"])
        Q      = read(bcrit["Scan"][ss]["Q"])
        balance = read(bcrit["Scan"][ss]["balance"])

        display(bcrit_diag_plots(delta, Q, balance, "surface_$i"))
    end
end

In [ ]:
p_eq = Analysis.Equilibrium.plot_equilibrium_summary(h5path)
p_ffs = Analysis.ForceFreeStates.plot_ffs_summary(h5path)

display(p_eq)
display(p_ffs)